# 1. Receita total de todos os filmes (R$)

In [0]:
%sql

SELECT
    SUM(receita_brl) AS receita_total_brl
FROM cinedata_medallion.gold.fact_movies_performance;

# 2. Os 5 filmes com maior popularidade

In [0]:
%sql
SELECT
    m.titulo,
    f.popularidade
FROM cinedata_medallion.gold.fact_movies_performance f
JOIN cinedata_medallion.gold.dim_movies m
    ON f.sk_movie_id = m.sk_movie_id
WHERE f.popularidade IS NOT NULL
ORDER BY f.popularidade DESC
LIMIT 5;

# 3. Quantidade de filmes por gênero

In [0]:
%sql
SELECT
    g.nome_genero,
    COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
FROM cinedata_medallion.gold.bridge_movie_genre b
JOIN cinedata_medallion.gold.dim_genres g
    ON b.sk_genre_id = g.sk_genre_id
GROUP BY g.nome_genero
ORDER BY qtd_filmes DESC;

# 4. Os 10 filmes de maior receita, com ranking

In [0]:
%sql
WITH ranking_filmes AS (
    SELECT
        m.titulo,
        f.receita_usd,
        f.receita_brl,
        RANK() OVER (
            ORDER BY f.receita_usd DESC
        ) AS ranking_receita
    FROM cinedata_medallion.gold.fact_movies_performance f
    JOIN cinedata_medallion.gold.dim_movies m
        ON f.sk_movie_id = m.sk_movie_id
    WHERE f.receita_usd IS NOT NULL
)
SELECT
    titulo,
    receita_usd,
    receita_brl,
    ranking_receita
FROM ranking_filmes
WHERE ranking_receita <= 10
ORDER BY ranking_receita;

# 5. Ator com mais participações nos filmes lançados nos últimos 2 anos

In [0]:
%sql
WITH data_referencia AS (
    SELECT
        MAX(data_lancamento) AS data_maxima
    FROM cinedata_medallion.gold.dim_movies
    WHERE data_lancamento IS NOT NULL
      AND data_lancamento <= CURRENT_DATE()
),
participacoes AS (
    SELECT
        p.sk_person_id,
        p.nome_pessoa,
        COUNT(DISTINCT b.sk_movie_id) AS qtd_participacoes
    FROM cinedata_medallion.gold.bridge_movie_person b
    JOIN cinedata_medallion.gold.dim_people p
        ON b.sk_person_id = p.sk_person_id
    JOIN cinedata_medallion.gold.dim_movies m
        ON b.sk_movie_id = m.sk_movie_id
    CROSS JOIN data_referencia d
    WHERE p.tipo_pessoa = 'Ator'
      AND m.data_lancamento >= ADD_MONTHS(d.data_maxima, -24)
      AND m.data_lancamento <= d.data_maxima
    GROUP BY
        p.sk_person_id,
        p.nome_pessoa
)
SELECT
    nome_pessoa,
    qtd_participacoes
FROM participacoes
ORDER BY qtd_participacoes DESC
LIMIT 1;

# 6. Produtora com maior lucro nos últimos 5 anos

In [0]:
%sql
WITH data_referencia AS (
    SELECT
        MAX(data_lancamento) AS data_maxima
    FROM cinedata_medallion.gold.dim_movies
    WHERE data_lancamento IS NOT NULL
      AND data_lancamento <= CURRENT_DATE()
),
lucro_produtoras AS (
    SELECT
        c.sk_company_id,
        c.nome_produtora,
        SUM(f.lucro_brl) AS lucro_total_brl,
        SUM(f.lucro_usd) AS lucro_total_usd
    FROM cinedata_medallion.gold.bridge_movie_company b
    JOIN cinedata_medallion.gold.dim_companies c
        ON b.sk_company_id = c.sk_company_id
    JOIN cinedata_medallion.gold.fact_movies_performance f
        ON b.sk_movie_id = f.sk_movie_id
    JOIN cinedata_medallion.gold.dim_movies m
        ON f.sk_movie_id = m.sk_movie_id
    CROSS JOIN data_referencia d
    WHERE m.data_lancamento >= ADD_MONTHS(d.data_maxima, -60)
      AND m.data_lancamento <= d.data_maxima
    GROUP BY
        c.sk_company_id,
        c.nome_produtora
)
SELECT
    nome_produtora,
    lucro_total_brl,
    lucro_total_usd
FROM lucro_produtoras
WHERE lucro_total_brl IS NOT NULL
ORDER BY lucro_total_brl DESC
LIMIT 1;